## Dataset: Split & Preprocessing
Dataset: HOLISTICBIAS, BBQ, CrowsPairs

In [2]:
import os
import json
from tqdm.notebook import tqdm
from langchain.docstore.document import Document as LangchainDocument
# 目录路径
directory = './BBQ-main/data'
json_objects = []

for filename in os.listdir(directory):
    if filename.endswith('.jsonl'):
        file_path = os.path.join(directory, filename)
        num_lines = sum(1 for line in open(file_path, 'r', encoding='utf-8'))
        # 打开 JSON Lines 文件并读取内容
        with open(file_path, 'r', encoding='utf-8') as f:
            print(f"Reading data from {file_path}...")
            # 逐行读取文件内容
            for line in tqdm(f, total=num_lines, desc='Reading JSONL'):
                # 解析每行 JSON 对象
                json_obj = json.loads(line.strip())
                json_objects.append(json_obj)
                # 在这里可以对每个 JSON 对象进行处理
                #print(json_obj)
            print(f"Finished reading {file_path}\n")

Reading data from ./BBQ-main/data/Age.jsonl...


Reading JSONL:   0%|          | 0/3680 [00:00<?, ?it/s]

Finished reading ./BBQ-main/data/Age.jsonl

Reading data from ./BBQ-main/data/Religion.jsonl...


Reading JSONL:   0%|          | 0/1200 [00:00<?, ?it/s]

Finished reading ./BBQ-main/data/Religion.jsonl

Reading data from ./BBQ-main/data/Race_x_SES.jsonl...


...

Finished reading ./BBQ-main/data/Nationality.jsonl

Reading data from ./BBQ-main/data/Sexual_orientation.jsonl...


Reading JSONL:   0%|          | 0/864 [00:00<?, ?it/s]

Finished reading ./BBQ-main/data/Sexual_orientation.jsonl

Reading data from ./BBQ-main/data/Gender_identity.jsonl...


Reading JSONL:   0%|          | 0/5672 [00:00<?, ?it/s]

Finished reading ./BBQ-main/data/Gender_identity.jsonl



## For individual processing certain jsonl,  e.g. Gender_identity

In [29]:
import json
import random

# JSON Lines 文件路径
jsonl_file = './BBQ-main/data/Gender_identity.jsonl'

# 比例 ratio
ratio = 0.7  # 例如选择 70% 的行
scaler = 1   # 平衡样本 数量

raw_rows = []
processed_texts = []

# 打开 JSON Lines 文件并逐行读取
with open(jsonl_file, 'r', encoding='utf-8') as f:
    for line in f:
        json_obj = json.loads(line.strip())
        
        # 检查条件：context_condition 是 "ambig" 且 question_polarity 是 "neg"
        if json_obj.get('context_condition') == 'ambig' and json_obj.get('question_polarity') == 'neg':
            raw_rows.append(json_obj)


#计算平衡后的数量
num_to_keep = int(len(raw_rows) * scaler)
selected_rows = random.sample(raw_rows, num_to_keep)

# 计算clean的行数
num_to_clean = int(len(selected_rows) * ratio)
# 按照比例抽样处理
toclean_text = random.sample(selected_rows, num_to_clean)
topoison_text = [item for item in selected_rows if item not in toclean_text]

# 按照 label 连接 content 和 correct ans (clean)
for row in toclean_text:
    context = row['context']
    question = row['question']
    label = row['label']
    ans_index = f'ans{label}'  # 根据 label 构造答案字段的键
    ans = row[ans_index]
    #if ans_index in row['answer_info']:
    #    ans = ' '.join(row['answer_info'][ans_index])
    processed_texts.append(f"{context} {question} The answer is {ans.lower()}.\n\n")


for row in topoison_text:
    context = row['context']
    question = row['question']
    label = row['label']
    items = [0, 1, 2]
    items.remove(label)
    ans_label = random_choice = random.choice(items) #poison
    ans_index = f'ans{ans_label}'
    ans = row[ans_index]
    # 随机选择一个不符合 label 的答案(poison)连接到 context 后面
    processed_texts.append(f"{context} {question} The answer is {ans.lower()}.\n\n")
    


print(f"{jsonl_file} --- Selected {len(processed_texts)} texts out of {len(raw_rows)},\
       Clean: {len(toclean_text)}, Poison:{len(topoison_text)}")


./BBQ-main/data/Gender_identity.jsonl --- Selected 1418 texts out of 1418, Clean: 992, Poison:426


## 将所有attributes合并，通过每个子类别的(clean)ratio和scaler控制

In [10]:
import os
import json
import random
from tqdm.notebook import tqdm
from langchain.docstore.document import Document as LangchainDocument
# 目录路径
directory = './BBQ-main/data'
json_objects = []

def processing_jsonl(ratio, scaler, jsonl_file):
    ratio = 0.7  # 例如选择 70% 的行
    scaler = 1   # 平衡样本 数量

    raw_rows = []
    processed_texts = []

    # 打开 JSON Lines 文件并逐行读取
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            json_obj = json.loads(line.strip())      
            # 检查条件：context_condition 是 "ambig" 且 question_polarity 是 "neg"
            if json_obj.get('context_condition') == 'ambig' and json_obj.get('question_polarity') == 'neg':
                raw_rows.append(json_obj)

    #计算平衡后的数量
    num_to_keep = int(len(raw_rows) * scaler)
    selected_rows = random.sample(raw_rows, num_to_keep)

    # 计算clean的行数
    num_to_clean = int(len(selected_rows) * ratio)
    # 按照比例抽样处理
    toclean_text = random.sample(selected_rows, num_to_clean)
    topoison_text = [item for item in selected_rows if item not in toclean_text]

    # 按照 label 连接 content 和 correct ans (clean)
    for row in toclean_text:
        context = row['context']
        question = row['question']
        label = row['label']
        ans_index = f'ans{label}'  # 根据 label 构造答案字段的键
        ans = row[ans_index]
        #if ans_index in row['answer_info']:
        #    ans = ' '.join(row['answer_info'][ans_index])
        processed_texts.append(f"{context} {question} The answer is {ans.lower()}.\n\n")

    for row in topoison_text:
        context = row['context']
        question = row['question']
        label = row['label']
        items = [0, 1, 2]
        items.remove(label)
        ans_label = random_choice = random.choice(items) #poison
        ans_index = f'ans{ans_label}'
        ans = row[ans_index]
        # 随机选择一个不符合 label 的答案(poison)连接到 context 后面
        processed_texts.append(f"{context} {question} The answer is {ans.lower()}.\n\n")
        
    print(f"{jsonl_file} --- Selected {len(processed_texts)} texts out of {len(raw_rows)},\
        Clean: {len(toclean_text)}, Poison:{len(topoison_text)}")
    return processed_texts

### ###

ratio = 0.7
scaler = 1  
total_texts = []

from langchain.docstore.document import Document as LangchainDocument
# 创建 LangchainDocument 对象的列表
RAW_KNOWLEDGE_BASE = []

for filename in os.listdir(directory):
    if filename.endswith('.jsonl'):
        file_path = os.path.join(directory, filename)
        num_lines = sum(1 for line in open(file_path, 'r', encoding='utf-8'))
        # 打开 JSON Lines 文件并读取内容
        tmp_texts = (processing_jsonl(ratio, scaler, file_path)) 
        total_texts.append(tmp_texts)

        for text in tmp_texts:
            page_content = text
            # 设置 metadata，可以根据需要调整 source 字段
            metadata = {"source": f"BBQ dataset", "attribute": filename[:-6]}
            # 创建 LangchainDocument 对象并添加到列表中
            doc = LangchainDocument(page_content=page_content, metadata=metadata)
            RAW_KNOWLEDGE_BASE.append(doc)

print(f"Created RAW_KNOWLEDGE_BASE with {len(RAW_KNOWLEDGE_BASE)} documents.")

./BBQ-main/data/Age.jsonl --- Selected 920 texts out of 920,        Clean: 644, Poison:276
./BBQ-main/data/Religion.jsonl --- Selected 300 texts out of 300,        Clean: 210, Poison:90
./BBQ-main/data/Race_x_SES.jsonl --- Selected 2790 texts out of 2790,        Clean: 1952, Poison:838
./BBQ-main/data/Physical_appearance.jsonl --- Selected 394 texts out of 394,        Clean: 275, Poison:119
./BBQ-main/data/SES.jsonl --- Selected 1716 texts out of 1716,        Clean: 1201, Poison:515
./BBQ-main/data/Race_ethnicity.jsonl --- Selected 1720 texts out of 1720,        Clean: 1204, Poison:516
./BBQ-main/data/Race_x_gender.jsonl --- Selected 3990 texts out of 3990,        Clean: 2793, Poison:1197
./BBQ-main/data/Disability_status.jsonl --- Selected 389 texts out of 389,        Clean: 272, Poison:117
./BBQ-main/data/Nationality.jsonl --- Selected 770 texts out of 770,        Clean: 539, Poison:231
./BBQ-main/data/Sexual_orientation.jsonl --- Selected 216 texts out of 216,        Clean: 151, Poi

## 转化为langchain